# Advanced Model Ensembling & Stacking - PS-S06E08

This notebook implements advanced ensembling, rank-scaling, and meta-model stacking for **Playground Series – Season 6, Episode 8: Predicting Smartphone Addiction**.

Ensembling & Stacking Strategies:
1. **Single Model Baseline Evaluation**
2. **Rank Normalization** (`scipy.stats.rankdata` percentile scaling)
3. **Caruana Hill-Climbing Selection** (Greedy ensemble selection with replacement)
4. **Optimized Weighted Probability Average** (Nelder-Mead simplex search)
5. **Multi-Model Meta-Stacking** (`RidgeCV`, `LogisticRegression`, `ExtraTrees`)

In [ ]:
import os
import glob
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold

from ps_s06e08_experiment_setup import ExperimentSetup
from ps_s06e08_model_visualizer import ModelVisualizer

warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
setup = ExperimentSetup(
    model_name="Ensemble",
    use_gpu=False,
    perform_rfe=False,
    perform_optuna_tuning=False
)

seed = setup.set_seeds()
setup.configure_pandas()
setup.suppress_warnings()

TARGET = "addicted_label"

In [ ]:
oof_files = []
test_files = []

if setup.running_in_kaggle():
    pred_dirs = ["/kaggle/input/notebooks/stephentarter/ps-s06e08-*/predictions"]
else:
    pred_dirs = ["predictions"]

EXCLUDE_PREFIXES = ("blending", "stacking", "ensemble")

def _is_base_model_file(path):
    name = os.path.basename(path)
    return not name.startswith(EXCLUDE_PREFIXES)

for p_dir in pred_dirs:
    oof_files.extend(glob.glob(f"{p_dir}/*_oof_probs.csv"))
    test_files.extend(glob.glob(f"{p_dir}/*_test_probs.csv"))

oof_files  = sorted(f for f in oof_files  if _is_base_model_file(f))
test_files = sorted(f for f in test_files if _is_base_model_file(f))

print(f"Found {len(oof_files)} OOF files and {len(test_files)} Test files.")
for f in oof_files:
    print(" -", os.path.basename(f))

In [ ]:
def load_probs(file_list, index_col="id"):
    df_list = []
    for file in file_list:
        base = os.path.basename(file)
        model_name = base.replace("_oof_probs.csv", "").replace("_test_probs.csv", "")
        df = pd.read_csv(file)
        prob_col = "prob_1"
        if prob_col not in df.columns:
            ignore = {"id", "target", TARGET}
            prob_cols = [c for c in df.columns if c not in ignore]
            if len(prob_cols) > 0:
                prob_col = prob_cols[-1]
            else:
                continue
        rename_map = {prob_col: model_name}
        subset = df.set_index(index_col).rename(columns=rename_map)[[model_name]]
        df_list.append(subset)
    return pd.concat(df_list, axis=1)

In [ ]:
training_df = setup.read_dataset("training")
train_ids = training_df["id"].copy()
y_true = training_df[TARGET].copy()

submission_df = setup.read_dataset("submission")
test_ids = submission_df["id"].copy()

oof_df = load_probs(oof_files) if oof_files else pd.DataFrame()
test_df = load_probs(test_files) if test_files else pd.DataFrame()

y_df = pd.DataFrame({"id": train_ids, "target": y_true}).set_index("id")
common_ids = y_df.index.intersection(oof_df.index)
if len(common_ids) == 0:
    raise RuntimeError("No overlapping IDs between training labels and OOF files.")

oof_df = oof_df.loc[common_ids].sort_index()
y = y_df.loc[oof_df.index, "target"].values

if not test_df.empty:
    test_df = test_df.loc[test_ids].copy()

def drop_bad_models(oof_df, test_df):
    bad = set()
    if not oof_df.empty:
        bad |= set(oof_df.columns[oof_df.isna().any(axis=0)])
    if not test_df.empty:
        bad |= set(test_df.columns[test_df.isna().any(axis=0)])
    if bad:
        print("Dropping models with NaNs:", sorted(bad))
        oof_df = oof_df.drop(columns=[c for c in bad if c in oof_df.columns], errors="ignore")
        test_df = test_df.drop(columns=[c for c in bad if c in test_df.columns], errors="ignore")
    common = [c for c in oof_df.columns if c in test_df.columns]
    return oof_df[common], test_df[common]

oof_df, test_df = drop_bad_models(oof_df, test_df)
model_names = list(oof_df.columns)

X_oof = oof_df.values
X_test = test_df.values

print(f"Aligned base models ({len(model_names)}): {model_names}")
print(f"OOF matrix shape: {X_oof.shape}")
print(f"Test matrix shape: {X_test.shape}")

In [ ]:
print("Single Model OOF ROC AUC scores:")
single_scores = {}
for i, name in enumerate(model_names):
    score = roc_auc_score(y, X_oof[:, i])
    single_scores[name] = score
    print(f"  {name}: {score:.6f}")

# Compute Percentile Rank Normalization
X_oof_rank = np.zeros_like(X_oof)
X_test_rank = np.zeros_like(X_test)
for i in range(len(model_names)):
    X_oof_rank[:, i] = rankdata(X_oof[:, i]) / len(X_oof)
    X_test_rank[:, i] = rankdata(X_test[:, i]) / len(X_test)

In [ ]:
def caruana_hill_climbing(X_oof_mat, y_true, names, max_iters=100):
    n_samples, n_models = X_oof_mat.shape
    single_aucs = [roc_auc_score(y_true, X_oof_mat[:, i]) for i in range(n_models)]
    best_idx = int(np.argmax(single_aucs))
    
    selected_indices = [best_idx]
    current_blend = X_oof_mat[:, best_idx].copy()
    current_best_auc = single_aucs[best_idx]
    
    print(f"Starting Hill-Climbing with best model: '{names[best_idx]}' (AUC: {current_best_auc:.6f})")
    
    for iteration in range(1, max_iters):
        best_candidate = None
        best_candidate_auc = current_best_auc
        
        for m in range(n_models):
            K = len(selected_indices)
            candidate_blend = (current_blend * K + X_oof_mat[:, m]) / (K + 1)
            candidate_auc = roc_auc_score(y_true, candidate_blend)
            
            if candidate_auc > best_candidate_auc:
                best_candidate_auc = candidate_auc
                best_candidate = m
                
        if best_candidate is not None:
            selected_indices.append(best_candidate)
            K = len(selected_indices)
            current_blend = (current_blend * (K - 1) + X_oof_mat[:, best_candidate]) / K
            current_best_auc = best_candidate_auc
        else:
            print(f"Hill-Climbing converged at iteration {iteration} with {len(selected_indices)} selections.")
            break
            
    counts = np.bincount(selected_indices, minlength=n_models)
    weights = counts / np.sum(counts)
    return weights, current_blend, current_best_auc

hill_weights, hill_oof_preds, hill_score = caruana_hill_climbing(X_oof_rank, y, model_names)
hill_test_preds = np.dot(X_test_rank, hill_weights)

print("\nCaruana Hill-Climbing Model Weights:")
for name, w in zip(model_names, hill_weights):
    print(f"  {name}: {w:.4f}")
print(f"Caruana Hill-Climbing OOF ROC AUC: {hill_score:.6f}")

In [ ]:
def loss_func(weights):
    w = weights / np.sum(weights)
    blend = np.dot(X_oof_rank, w)
    return -roc_auc_score(y, blend)

res = minimize(loss_func, [1/len(model_names)]*len(model_names), method="Nelder-Mead", bounds=[(0, 1)]*len(model_names))
opt_weights = res.x / np.sum(res.x)
opt_blend_oof = np.dot(X_oof_rank, opt_weights)
opt_blend_test = np.dot(X_test_rank, opt_weights)
opt_score = roc_auc_score(y, opt_blend_oof)

print("Nelder-Mead Rank-Weighted Average OOF ROC AUC:", f"{opt_score:.6f}")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# 1. RidgeCV Stacking
ridge_oof = np.zeros(len(y))
ridge_test = np.zeros(len(X_test))
for tr_idx, val_idx in skf.split(X_oof_rank, y):
    m = RidgeCV(alphas=np.logspace(-3, 3, 20))
    m.fit(X_oof_rank[tr_idx], y[tr_idx])
    ridge_oof[val_idx] = m.predict(X_oof_rank[val_idx])
    ridge_test += m.predict(X_test_rank) / 5.0
ridge_score = roc_auc_score(y, ridge_oof)
print(f"RidgeCV Stacking OOF ROC AUC: {ridge_score:.6f}")

# 2. LogisticRegression Stacking
logreg_oof = np.zeros(len(y))
logreg_test = np.zeros(len(X_test))
for tr_idx, val_idx in skf.split(X_oof_rank, y):
    m = LogisticRegression(C=0.1, solver="lbfgs")
    m.fit(X_oof_rank[tr_idx], y[tr_idx])
    logreg_oof[val_idx] = m.predict_proba(X_oof_rank[val_idx])[:, 1]
    logreg_test += m.predict_proba(X_test_rank)[:, 1] / 5.0
logreg_score = roc_auc_score(y, logreg_oof)
print(f"LogisticRegression Stacking OOF ROC AUC: {logreg_score:.6f}")

# 3. ExtraTrees Meta Stacking
et_oof = np.zeros(len(y))
et_test = np.zeros(len(X_test))
for tr_idx, val_idx in skf.split(X_oof_rank, y):
    m = ExtraTreesClassifier(n_estimators=200, max_depth=3, random_state=seed, n_jobs=-1)
    m.fit(X_oof_rank[tr_idx], y[tr_idx])
    et_oof[val_idx] = m.predict_proba(X_oof_rank[val_idx])[:, 1]
    et_test += m.predict_proba(X_test_rank)[:, 1] / 5.0
et_score = roc_auc_score(y, et_oof)
print(f"ExtraTrees Stacking OOF ROC AUC: {et_score:.6f}")

In [ ]:
simple_avg_oof = np.mean(X_oof_rank, axis=1)
simple_avg_test = np.mean(X_test_rank, axis=1)
simple_score = roc_auc_score(y, simple_avg_oof)

best_single_name = max(single_scores, key=single_scores.get)
best_single_score = single_scores[best_single_name]

results = [
    {"Strategy": f"Best Single Model ({best_single_name})", "OOF ROC AUC": best_single_score, "OOF_Preds": X_oof_rank[:, model_names.index(best_single_name)], "Test_Preds": X_test_rank[:, model_names.index(best_single_name)]},
    {"Strategy": "Simple Rank Average", "OOF ROC AUC": simple_score, "OOF_Preds": simple_avg_oof, "Test_Preds": simple_avg_test},
    {"Strategy": "Caruana Hill-Climbing", "OOF ROC AUC": hill_score, "OOF_Preds": hill_oof_preds, "Test_Preds": hill_test_preds},
    {"Strategy": "Nelder-Mead Rank-Weighted Average", "OOF ROC AUC": opt_score, "OOF_Preds": opt_blend_oof, "Test_Preds": opt_blend_test},
    {"Strategy": "RidgeCV Stacking", "OOF ROC AUC": ridge_score, "OOF_Preds": ridge_oof, "Test_Preds": ridge_test},
    {"Strategy": "LogisticRegression Stacking", "OOF ROC AUC": logreg_score, "OOF_Preds": logreg_oof, "Test_Preds": logreg_test},
    {"Strategy": "ExtraTrees Stacking", "OOF ROC AUC": et_score, "OOF_Preds": et_oof, "Test_Preds": et_test},
]

df_results = pd.DataFrame(results).sort_values("OOF ROC AUC", ascending=False).reset_index(drop=True)
print("\n================ ENSEMBLE PERFORMANCE SUMMARY ================")
for idx, row in df_results.iterrows():
    strat_name = row["Strategy"]
    score = row["OOF ROC AUC"]
    print(f"{idx+1}. {strat_name:<42}: {score:.6f}")
print("="*63)

winning_row = df_results.iloc[0]
print(f"\nWinning Ensemble Strategy: {winning_row['Strategy']} (OOF ROC AUC: {winning_row['OOF ROC AUC']:.6f})")

# Save final submission.csv
final_test_probs = winning_row["Test_Preds"]
final_test_ranks = rankdata(final_test_probs) / len(final_test_probs)

sub = pd.DataFrame({
    "id": test_ids,
    "addicted_label": final_test_ranks
})
sub.to_csv("submission.csv", index=False)
print("Saved final submission.csv with rank-normalized probabilities successfully!")